# Class Imbalance & Cost-Sensitive Threshold Optimisation

The Telco dataset has ~26 % churn — mild but enough to bias classifiers toward the
majority class if left uncorrected.  More importantly, the two error types carry
asymmetric costs:

| Error | Description | Assumed cost |
|---|---|---|
| False Negative (FN) | Churner not flagged; customer leaves | **$500** (lost CLV) |
| False Positive (FP) | Loyal customer receives unnecessary offer | **$50** (offer + handling) |

The default 0.5 decision threshold optimises accuracy, not business value.  This
notebook answers two questions:
1. Which imbalance strategy (none / class weights / SMOTE) best recovers churners?
2. At what probability threshold does the business cost reach its minimum?

| Section | Content |
|---|---|
| 1. Setup | Imports, data, config |
| 2. Imbalance strategies | CV comparison: none vs class_weight vs SMOTE |
| 3. PR curves | Precision-recall curves per strategy |
| 4. Cost curve | Expected cost vs threshold (tuned XGBoost) |
| 5. Sensitivity analysis | How optimal threshold shifts with cost ratio |
| 6. Summary | Recommendation and next steps |

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.load import download_telco_data
from src.data.pipeline import prepare_data
from src.evaluation.threshold import (
    cost_curve,
    optimal_threshold,
    threshold_sensitivity,
)
from src.models.imbalance import compare_imbalance_strategies
from src.utils.plotting import PALETTE, save_fig, set_plot_style

set_plot_style()
FIGURES_DIR = REPO_ROOT / "reports" / "figures"

In [ ]:
with open(REPO_ROOT / "configs" / "config.yaml") as fh:
    cfg = yaml.safe_load(fh)

SEED: int = cfg["random_seed"]
RAW_PATH = REPO_ROOT / cfg["paths"]["raw_data"]
COST_FN: float = cfg["cost_matrix"]["false_negative"]
COST_FP: float = cfg["cost_matrix"]["false_positive"]

download_telco_data(RAW_PATH, urls=cfg["data"]["download_urls"])

data = prepare_data(
    RAW_PATH,
    val_size=cfg["data"]["val_size"],
    test_size=cfg["data"]["test_size"],
    seed=SEED,
    processed_dir=REPO_ROOT / cfg["paths"]["processed_data"],
)

X_train = data["X_train"]
y_train = data["y_train"].to_numpy()
X_val   = data["X_val"]
y_val   = data["y_val"].to_numpy()
X_test  = data["X_test"]
y_test  = data["y_test"].to_numpy()

X_cv = np.vstack([X_train, X_val])
y_cv = np.concatenate([y_train, y_val])

print(f"Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}")
print(f"Churn rate — train: {y_train.mean():.1%}  val: {y_val.mean():.1%}  test: {y_test.mean():.1%}")
print(f"Cost matrix — FN: ${COST_FN:.0f}  FP: ${COST_FP:.0f}  ratio: {COST_FN/COST_FP:.0f}:1")

## 1. Imbalance Strategy Comparison

Three strategies are evaluated on a Logistic Regression base model using
5-fold stratified CV on the combined train+val pool.  LR is chosen because
the effect of class-imbalance correction is most transparent on linear
classifiers — if SMOTE or class weights help materially, they will also
help GBMs.

**Why these three?**
- **No correction**: baseline; classifier assigns equal importance to all samples.
- **`class_weight='balanced'`**: re-weights loss by inverse class frequency.
  Zero additional computation; no synthetic samples.
- **SMOTE**: over-samples the minority class by interpolation.  Applied
  *inside each CV fold* via imblearn Pipeline to prevent data leakage.

In [ ]:
comparison, val_probas = compare_imbalance_strategies(
    X_train, y_train, X_val, y_val, seed=SEED, n_splits=5
)

display_cols = [
    "strategy",
    "cv_auc_mean", "cv_auc_std",
    "cv_recall_mean", "cv_recall_std",
    "cv_avg_precision_mean", "cv_avg_precision_std",
    "val_auc", "val_recall", "val_avg_precision",
]

fmt = {
    col: "{:.4f}" for col in display_cols if col != "strategy"
}

display(
    comparison[display_cols]
    .style.format(fmt)
    .background_gradient(subset=["cv_auc_mean", "cv_recall_mean"], cmap="RdYlGn")
    .set_caption("5-fold CV + held-out validation metrics per imbalance strategy (LR base model)")
)

In [ ]:
# Bar chart: CV recall and AUC side by side
strategies = comparison["strategy"].tolist()
x = np.arange(len(strategies))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
bars = ax.bar(
    x, comparison["cv_recall_mean"], width,
    color=PALETTE[:3], edgecolor="white", alpha=0.9
)
ax.errorbar(
    x, comparison["cv_recall_mean"], yerr=comparison["cv_recall_std"],
    fmt="none", color="black", capsize=4, linewidth=1.2
)
ax.set_xticks(x)
ax.set_xticklabels(strategies, rotation=15, ha="right")
ax.set_ylabel("CV Recall (mean ± std)")
ax.set_title("Recall by Imbalance Strategy")
ax.set_ylim(0, 1.0)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.3f}", ha="center", fontsize=9)

ax = axes[1]
bars = ax.bar(
    x, comparison["cv_avg_precision_mean"], width,
    color=PALETTE[:3], edgecolor="white", alpha=0.9
)
ax.errorbar(
    x, comparison["cv_avg_precision_mean"], yerr=comparison["cv_avg_precision_std"],
    fmt="none", color="black", capsize=4, linewidth=1.2
)
ax.set_xticks(x)
ax.set_xticklabels(strategies, rotation=15, ha="right")
ax.set_ylabel("CV Average Precision (PR-AUC)")
ax.set_title("PR-AUC by Imbalance Strategy")
ax.set_ylim(0, 1.0)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.3f}", ha="center", fontsize=9)

fig.suptitle("Imbalance Strategy Comparison (LR, 5-fold CV)", y=1.02)
fig.tight_layout()
save_fig(fig, "07_imbalance_comparison", FIGURES_DIR)
plt.show()

## 2. Precision-Recall Curves by Strategy

PR curves are more informative than ROC curves when one class is rare: ROC
remains high even when the model assigns most positives incorrectly, because
the large TN pool inflates the FPR denominator.  Average Precision (area
under the PR curve) is therefore a better summary for the business problem.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, average_precision_score

strategy_keys = ["none", "class_weight", "smote"]
strategy_labels_map = {
    "none": "No balancing",
    "class_weight": "class_weight='balanced'",
    "smote": "SMOTE",
}

fig, ax = plt.subplots(figsize=(7, 6))

for key, color in zip(strategy_keys, PALETTE):
    proba = val_probas[key]
    ap = average_precision_score(y_val, proba)
    PrecisionRecallDisplay.from_predictions(
        y_val, proba, ax=ax,
        name=f"{strategy_labels_map[key]} (AP={ap:.3f})",
        color=color,
    )

churn_prevalence = y_val.mean()
ax.axhline(
    churn_prevalence, linestyle="--", linewidth=0.8, color="grey",
    label=f"Random classifier (AP={churn_prevalence:.3f})",
)
ax.set_title("Precision-Recall Curves — Imbalance Strategies (Validation Set)")
ax.legend(loc="upper right", frameon=False)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
fig.tight_layout()
save_fig(fig, "07_pr_curves_strategies", FIGURES_DIR)
plt.show()

## 3. Cost Curve — Tuned XGBoost

The cost curve replaces the confusion-matrix snapshot at an arbitrary 0.5
threshold with a full view of how expected cost changes across *all* thresholds.
The optimal threshold is the point that minimises:

$$\text{expected\_cost}(t) = FN(t) \times C_{FN} + FP(t) \times C_{FP}$$

With $C_{FN} = \$500$ and $C_{FP} = \$50$ (10:1 ratio).

In [ ]:
# Train XGBoost with default params for the cost analysis
from xgboost import XGBClassifier

xgb_cfg = cfg.get("xgboost", {})

model_xgb = XGBClassifier(
    n_estimators=xgb_cfg.get("n_estimators", 300),
    learning_rate=xgb_cfg.get("learning_rate", 0.05),
    max_depth=xgb_cfg.get("max_depth", 6),
    subsample=xgb_cfg.get("subsample", 0.8),
    colsample_bytree=xgb_cfg.get("colsample_bytree", 0.8),
    eval_metric="logloss",
    random_state=SEED,
    verbosity=0,
)
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=50,
    verbose=False,
)

# Evaluate on val set for threshold optimisation
val_proba_xgb = model_xgb.predict_proba(X_val)[:, 1]

best_t, df_cost = optimal_threshold(
    y_val, val_proba_xgb, cost_fn=COST_FN, cost_fp=COST_FP
)

default_row = df_cost.iloc[(df_cost["threshold"] - 0.5).abs().argsort()[:1]].iloc[0]
best_row = df_cost[df_cost["threshold"] == best_t].iloc[0]

print(f"Cost ratio          : FN/FP = {COST_FN/COST_FP:.0f}:1")
print(f"Default threshold   : 0.50   cost=${default_row['expected_cost']:,.0f}  "
      f"recall={default_row['recall']:.3f}  precision={default_row['precision']:.3f}")
print(f"Optimal threshold   : {best_t:.2f}   cost=${best_row['expected_cost']:,.0f}  "
      f"recall={best_row['recall']:.3f}  precision={best_row['precision']:.3f}")
print(f"Cost reduction      : ${default_row['expected_cost'] - best_row['expected_cost']:,.0f} "
      f"({(1 - best_row['expected_cost']/default_row['expected_cost']):.1%})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full cost curve
ax = axes[0]
ax.plot(df_cost["threshold"], df_cost["expected_cost"], color=PALETTE[0], linewidth=1.8)
ax.axvline(best_t, color=PALETTE[1], linestyle="--", linewidth=1.4,
           label=f"Optimal: t={best_t:.2f}  cost=${best_row['expected_cost']:,.0f}")
ax.axvline(0.5, color="grey", linestyle=":", linewidth=1.2,
           label=f"Default: t=0.50  cost=${default_row['expected_cost']:,.0f}")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Expected cost ($)")
ax.set_title(f"Cost Curve  (FN=${COST_FN:.0f}, FP=${COST_FP:.0f})")
ax.legend(frameon=False)

# Right: recall and precision vs threshold
ax = axes[1]
ax.plot(df_cost["threshold"], df_cost["recall"], color=PALETTE[0],
        linewidth=1.8, label="Recall (churners caught)")
ax.plot(df_cost["threshold"], df_cost["precision"], color=PALETTE[1],
        linewidth=1.8, label="Precision")
ax.axvline(best_t, color="black", linestyle="--", linewidth=1.2,
           label=f"Optimal t={best_t:.2f}")
ax.axvline(0.5, color="grey", linestyle=":", linewidth=1.0, label="Default t=0.50")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Rate")
ax.set_title("Recall & Precision vs Threshold")
ax.legend(frameon=False)

fig.suptitle("Cost-Sensitive Threshold Optimisation — XGBoost on Validation Set", y=1.01)
fig.tight_layout()
save_fig(fig, "07_cost_curve", FIGURES_DIR)
plt.show()

## 4. Confusion Matrix: Default vs Optimal Threshold

Comparing what the decision matrices look like at 0.5 vs the optimal threshold
makes the business impact concrete.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

preds_default = (val_proba_xgb >= 0.5).astype(int)
preds_optimal = (val_proba_xgb >= best_t).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title, threshold in zip(
    axes,
    [preds_default, preds_optimal],
    ["Default threshold (0.50)", f"Optimal threshold ({best_t:.2f})"],
    [0.5, best_t],
):
    ConfusionMatrixDisplay.from_predictions(
        y_val, preds,
        display_labels=["No churn", "Churn"],
        cmap="Blues",
        ax=ax,
        colorbar=False,
    )
    fp = int(((preds == 1) & (y_val == 0)).sum())
    fn = int(((preds == 0) & (y_val == 1)).sum())
    cost = fn * COST_FN + fp * COST_FP
    ax.set_title(f"{title}\nFN={fn}  FP={fp}  total cost=${cost:,.0f}")

fig.suptitle("Confusion Matrices — XGBoost on Validation Set", y=1.02)
fig.tight_layout()
save_fig(fig, "07_confusion_default_vs_optimal", FIGURES_DIR)
plt.show()

## 5. Sensitivity to Cost Ratio Assumption

The 10:1 cost ratio is an assumption, not a measurement.  The sensitivity
analysis sweeps the ratio from 1:1 to 50:1 and shows how the optimal
threshold responds.  A stable region (flat line) means the recommendation
is robust to reasonable uncertainty in the cost estimates.

In [ ]:
sens_df = threshold_sensitivity(
    y_val, val_proba_xgb,
    cost_fp=COST_FP,
    fn_fp_ratios=[1, 2, 3, 5, 7, 10, 15, 20, 30, 50],
)

display(
    sens_df.style
    .format({
        "cost_fn": "${:.0f}",
        "optimal_threshold": "{:.3f}",
        "recall_at_opt": "{:.3f}",
        "precision_at_opt": "{:.3f}",
        "expected_cost": "${:,.0f}",
    })
    .background_gradient(subset=["optimal_threshold"], cmap="RdYlGn_r")
    .set_caption("Optimal threshold and metrics at each FN/FP cost ratio")
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: optimal threshold vs cost ratio
ax = axes[0]
ax.plot(
    sens_df["fn_fp_ratio"], sens_df["optimal_threshold"],
    color=PALETTE[0], linewidth=2.0, marker="o", markersize=6,
)
ax.axvline(
    COST_FN / COST_FP, color=PALETTE[1], linestyle="--", linewidth=1.2,
    label=f"Assumed ratio ({COST_FN/COST_FP:.0f}:1)",
)
ax.set_xlabel("FN/FP cost ratio")
ax.set_ylabel("Optimal decision threshold")
ax.set_title("Optimal Threshold vs Cost Ratio")
ax.legend(frameon=False)

# Right: recall and precision at the optimal threshold
ax = axes[1]
ax.plot(
    sens_df["fn_fp_ratio"], sens_df["recall_at_opt"],
    color=PALETTE[0], linewidth=2.0, marker="o", markersize=6, label="Recall",
)
ax.plot(
    sens_df["fn_fp_ratio"], sens_df["precision_at_opt"],
    color=PALETTE[1], linewidth=2.0, marker="s", markersize=6, label="Precision",
)
ax.axvline(
    COST_FN / COST_FP, color="grey", linestyle="--", linewidth=1.0,
    label=f"Assumed ratio ({COST_FN/COST_FP:.0f}:1)",
)
ax.set_xlabel("FN/FP cost ratio")
ax.set_ylabel("Rate at optimal threshold")
ax.set_title("Recall & Precision at Optimal Threshold")
ax.legend(frameon=False)

fig.suptitle("Sensitivity to Cost Ratio Assumption", y=1.01)
fig.tight_layout()
save_fig(fig, "07_threshold_sensitivity", FIGURES_DIR)
plt.show()

## 6. Apply Optimal Threshold to Test Set

Apply the threshold found on the validation set to the held-out test set.
The test set has not been used in any decision so far — this is a clean
estimate of live deployment performance.

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

test_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

for label, threshold in [("Default (0.50)", 0.5), (f"Optimal ({best_t:.2f})", best_t)]:
    preds = (test_proba_xgb >= threshold).astype(int)
    fp = int(((preds == 1) & (y_test == 0)).sum())
    fn = int(((preds == 0) & (y_test == 1)).sum())
    cost = fn * COST_FN + fp * COST_FP
    print(f"{label:25s}  "
          f"AUC={roc_auc_score(y_test, test_proba_xgb):.4f}  "
          f"F1={f1_score(y_test, preds, zero_division=0):.4f}  "
          f"Recall={recall_score(y_test, preds, zero_division=0):.4f}  "
          f"Prec={precision_score(y_test, preds, zero_division=0):.4f}  "
          f"FN={fn}  FP={fp}  Cost=${cost:,.0f}")

## Summary

### Imbalance strategy

With ~26 % churn, all three strategies (none / class_weight / SMOTE) produce
similar ROC-AUC.  The key differentiator is **recall and PR-AUC**:

- **No correction**: highest precision, but misses more churners.
- **`class_weight='balanced'`**: best recall with no computational overhead.
  Recommended for all downstream models.
- **SMOTE**: statistically indistinguishable from class weighting on this
  dataset.  The ~26 % minority proportion is not extreme enough for synthetic
  over-sampling to add meaningful signal beyond what re-weighting achieves.

**Decision**: use `class_weight='balanced'` (or `scale_pos_weight` for GBMs)
as the standard imbalance correction.  Reserve SMOTE for datasets where
churn prevalence drops below ~10 %.

### Threshold optimisation

The 10:1 cost ratio (FN costs 10× more than FP) shifts the optimal threshold
well below 0.5.  The sensitivity analysis shows the threshold is stable in
the 7–15× range — the expected operating window for this business.

**Decision**: deploy with the cost-optimised threshold, not the default 0.5.
Recalibrate quarterly when CLV estimates or offer costs change.  Use
`src/evaluation/threshold.optimal_threshold()` to recompute programmatically.